In [ ]:
# 시나리오 1 — pandas + dtype 최적화 (전체 분석을 한 함수로 묶음)
def analyze_pandas_optimized(csv_path):
    dtype_map = {
        "log_id": "int32", "session_id": "int32",
        "page": "category", "device": "category",
        "status_code": "int16", "response_ms": "int16", "bytes_sent": "int32",
    }
    df = pd.read_csv(csv_path, dtype=dtype_map, parse_dates=["ts"])

    # (1) 페이지별 응답 시간 분포 — 평균·중앙값·표준편차
    by_page = df.groupby("page", observed=True)["response_ms"].agg(
        ["mean", "median", "std"]
    ).round(2)

    # (2) 디바이스별 트래픽 점유율 — 총 bytes_sent의 디바이스별 비율
    dev_total = df.groupby("device", observed=True)["bytes_sent"].sum()
    dev_share = (dev_total / dev_total.sum() * 100).round(1).rename("share_pct")

    # (3) 에러 발생 패턴 — status_code 400+ 의 페이지별 카운트
    err_count = (df[df["status_code"] >= 400]
                 .groupby("page", observed=True).size()
                 .rename("n_errors").sort_values(ascending=False))

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_pd = analyze_pandas_optimized(logs_csv)
elapsed_pd_full = time.perf_counter() - t0
peak_pd_full = rss_mb() - before

print(f"[pandas + dtype] 소요 시간: {elapsed_pd_full:.2f}초, 메모리 증가량: {peak_pd_full:.1f} MB\n")
print("(1) 페이지별 응답 시간"); print(res_pd["by_page"]); print()
print("(2) 디바이스별 트래픽 점유율(%)"); print(res_pd["dev_share"]); print()
print("(3) 에러 카운트"); print(res_pd["err_count"])

[pandas + dtype] 소요 시간: 2.33초, 메모리 증가량: 2.7 MB

(1) 페이지별 응답 시간
            mean  median     std
page                            
cart      160.27   134.0  113.13
checkout  159.94   134.0  112.94
detail    160.03   135.0  113.00
home      159.86   134.0  112.79
list      159.97   134.0  112.89
mypage    158.34   132.0  111.65
search    160.42   134.0  113.82

(2) 디바이스별 트래픽 점유율(%)
device
desktop    25.1
mobile     69.9
tablet      4.9
Name: share_pct, dtype: float64

(3) 에러 카운트
page
home        42898
list        35593
detail      28298
cart        14360
search       7256
mypage       7244
checkout     7108
Name: n_errors, dtype: int64

In [ ]:
# 시나리오 2 — 청크 처리 (같은 분석을 50K 청크로)
def analyze_chunked(csv_path, chunksize=50_000):
    # 누적 컨테이너
    sum_ms = {}; cnt_ms = {}; ms_lists = {}
    dev_bytes = {}; err_by_page = {}

    for chunk in pd.read_csv(csv_path, chunksize=chunksize,
                              dtype={"page": "category", "device": "category",
                                     "status_code": "int16", "response_ms": "int16"},
                              parse_dates=["ts"]):
        # 페이지별 합계·개수 (평균/표준편차 재계산용으로 모든 값 모음)
        for page, g in chunk.groupby("page", observed=True):
            sum_ms[page] = sum_ms.get(page, 0) + g["response_ms"].sum()
            cnt_ms[page] = cnt_ms.get(page, 0) + g["response_ms"].count()
            ms_lists.setdefault(page, []).append(g["response_ms"].values)

        # 디바이스별 bytes 합
        for dev, g in chunk.groupby("device", observed=True):
            dev_bytes[dev] = dev_bytes.get(dev, 0) + g["bytes_sent"].sum()

        # 에러 페이지별 카운트
        errs = chunk[chunk["status_code"] >= 400].groupby("page", observed=True).size()
        for page, n in errs.items():
            err_by_page[page] = err_by_page.get(page, 0) + n

    # reduce — 페이지별 통계 (mean/median/std)
    by_page = pd.DataFrame({
        page: {
            "mean":   sum_ms[page] / cnt_ms[page],
            "median": float(np.median(np.concatenate(ms_lists[page]))),
            "std":    float(np.concatenate(ms_lists[page]).std()),
        } for page in sum_ms
    }).T.round(2)

    dev_total = pd.Series(dev_bytes)
    dev_share = (dev_total / dev_total.sum() * 100).round(1).rename("share_pct")
    err_count = pd.Series(err_by_page).sort_values(ascending=False).rename("n_errors")

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_chunk = analyze_chunked(logs_csv)
elapsed_chunk_full = time.perf_counter() - t0
peak_chunk_full = rss_mb() - before

print(f"[chunked] 소요 시간: {elapsed_chunk_full:.2f}초, 메모리 증가량: {peak_chunk_full:.1f} MB")
print("by_page (chunked):"); print(res_chunk["by_page"].sort_index())

[chunked] 소요 시간: 2.37초, 메모리 증가량: 0.2 MB
by_page (chunked):
            mean  median     std
cart      160.27   134.0  113.13
checkout  159.94   134.0  112.94
detail    160.03   135.0  113.00
home      159.86   134.0  112.79
list      159.97   134.0  112.89
mypage    158.34   132.0  111.65
search    160.42   134.0  113.81

In [ ]:
# 시나리오 3 — Polars lazy (한 파이프라인)
def analyze_polars_lazy(csv_path):
    lf = pl.scan_csv(csv_path, try_parse_dates=True)

    # (1) 페이지별 응답 시간 — mean/median/std 한 번에
    by_page_lf = lf.group_by("page").agg([
        pl.col("response_ms").mean().alias("mean"),
        pl.col("response_ms").median().alias("median"),
        pl.col("response_ms").std().alias("std"),
    ]).sort("page")

    # (2) 디바이스별 점유율
    dev_lf = lf.group_by("device").agg(
        pl.col("bytes_sent").sum().alias("total_bytes")
    )
    # 점유율은 collect 후에 계산 (전체 합 필요)

    # (3) 에러 페이지별 카운트 — Predicate pushdown 적용 예상
    err_lf = (
        lf.filter(pl.col("status_code") >= 400)
        .group_by("page")
        .agg(pl.len().alias("n_errors"))
        .sort("n_errors", descending=True)
    )

    by_page = by_page_lf.collect().to_pandas().set_index("page").round(2)
    dev_tot = dev_lf.collect().to_pandas().set_index("device")["total_bytes"]
    dev_share = (dev_tot / dev_tot.sum() * 100).round(1).rename("share_pct")
    err_count = err_lf.collect().to_pandas().set_index("page")["n_errors"]

    return {"by_page": by_page, "dev_share": dev_share, "err_count": err_count}

gc.collect()
before = rss_mb()
t0 = time.perf_counter()
res_pl = analyze_polars_lazy(logs_csv)
elapsed_pl_full = time.perf_counter() - t0
peak_pl_full = rss_mb() - before

print(f"[Polars lazy] 소요 시간: {elapsed_pl_full:.2f}초, 메모리 증가량: {peak_pl_full:.1f} MB\n")
print("by_page (polars):"); print(res_pl["by_page"].sort_index())

[Polars lazy] 소요 시간: 0.32초, 메모리 증가량: 2.4 MB

by_page (polars):
            mean  median     std
page                            
cart      160.27   134.0  113.13
checkout  159.94   134.0  112.94
detail    160.03   135.0  113.00
home      159.86   134.0  112.79
list      159.97   134.0  112.89
mypage    158.34   132.0  111.65
search    160.42   134.0  113.82

# 퀴즈
1. pandas가 메모리에 데이터를 다 못 올릴 때 가장 먼저 시도해볼 두 가지 전략은 무엇인가요?전략 1: chunksize 옵션을 사용하여 쪼개 읽기 (pd.read_csv(..., chunksize=N))전체 데이터를 한 번에 올리지 않고, 내 메모리가 버틸 수 있는 크기(예: 5만 행씩)로 조각내어 순차적으로 처리하는 방법입니다.전략 2: 데이터 타입 다이어트 (다운캐스팅 및 category 활용)기본적으로 아주 크게 잡히는 타입들(예: float64, int64, object 문자열)을 더 작은 타입(float32, int8, category 등)으로 변환하여 메모리 사용량을 50%에서 80% 이상 절약하는 방법입니다.

2. int8로 표현 가능한 값의 범위는 어떻게 되나요?정답: (a) -128 ~ 127설명: 컴퓨터는 2진수를 쓰며, 8비트(int8)는 $2^8 = 256$개의 숫자를 표현할 수 있습니다. 음수와 양수를 반씩 나누어 가져가기 때문에 -128부터 127까지 표현하게 됩니다. (참고로 부호가 없는 8비트 정수형인 uint8이 0~255 범위인 (b)번에 해당합니다.)

3. 청크 처리에서 평균을 합칠 때 "평균의 평균"이 틀린 이유는 무엇인가요?정답: 각 청크에 포함된 데이터 개수(가중치)가 서로 다를 수 있기 때문입니다.설명: 1번 청크는 데이터가 1건(값: 10)이고, 2번 청크는 데이터가 99건(값 모두 100)일 때, 둘의 평균을 단순히 더해서 나누면 대다수 데이터의 무게(가중치)가 무시되어 평균이 심하게 왜곡됩니다. 따라서 반드시 각 청크에서 '합계(Sum)'와 '개수(Count)'를 각각 누적한 뒤 마지막에 나누어야 올바른 평균을 구할 수 있습니다.

4. Polars의 lazy 모드 시작 함수는 pl.read_csv 일까요, pl.scan_csv 일까요?정답: pl.scan_csv설명: pl.read_csv는 pandas처럼 즉시 데이터를 메모리에 올리는 Eager 모드 함수입니다. 반면 pl.scan_csv는 데이터를 바로 가져오지 않고 "이렇게 계산하겠다"는 계획만 세우는 Lazy 모드를 시작하는 열쇠입니다.

5. 데이터가 1억 행이고 매일 자동 실행 파이프라인이라면 첫 번째로 추천할 도구는 무엇인가요?정답: Polars (특히 Lazy 모드)설명: 1억 행은 pandas로 처리하기에는 메모리가 터질 확률이 매우 높고 속도도 지나치게 느립니다. 매일 자동화되어 실행되는 배치 파이프라인이라면, 모든 CPU 코어를 풀가동하고 Predicate Pushdown 같은 자체 최적화 기능으로 처리 속도와 메모리를 최소화해 주는 Polars Lazy가 압도적인 원탑 추천 도구입니다.

# 코드 퀴즈
logs_csv에서 device == 'mobile' 이고 status_code == 200 인 행만 골라, page별 response_ms의 95-percentile 을 구하세요. Polars lazy 모드 로 작성합니다.

In [ ]:
# 코드 퀴즈 — 모범 답안
ans = (
    pl.scan_csv(logs_csv)
    .filter((pl.col("device") == "mobile") & (pl.col("status_code") == 200))
    .group_by("page")
    .agg(pl.col("response_ms").quantile(0.95).alias("p95_ms"))
    .sort("p95_ms", descending=True)
    .collect()
)
print(ans)

shape: (7, 2)
┌──────────┬────────┐
│ page     ┆ p95_ms │
│ ---      ┆ ---    │
│ str      ┆ f64    │
╞══════════╪════════╡
│ checkout ┆ 387.0  │
│ cart     ┆ 381.0  │
│ mypage   ┆ 381.0  │
│ search   ┆ 380.0  │
│ home     ┆ 379.0  │
│ list     ┆ 378.0  │
│ detail   ┆ 375.0  │
└──────────┴────────┘
읽는_법: (조건1) & (조건2) 처럼 두 조건을 묶을 때는 각 조건을 괄호로 감싸 야 합니다. quantile(0.95)는 95번째 백분위수 — "응답 시간이 이보다 빠른 요청이 95%" 라는 뜻이에요. 페이지별로 상위 5% 느린 요청 의 응답 시간을 볼 수 있습니다.